# OLOG Data Retriever

In [1]:
import matplotlib.pyplot as plt
import requests
import pandas as pd
import xml.etree.ElementTree as ET

In [47]:
url = "https://ais-eng-srv-ca.cnpem.br/Olog/resources/logs"

paramaters = {
    "search": "*terremoto*",
    "sort": "created",
    "page": 1,
    "limit": 400,
}

response = requests.get(url, params=paramaters)

df = pd.json_normalize(response.json())

df = df.reindex(["id", "version", "owner", "createdDate", "eventStart", "description", "attachments"],axis=1)

paramaters = {
    "search": "*sísmico*",
    "sort": "created",
    "page": 1,
    "limit": 400,
}

response = requests.get(url, params=paramaters)

df_2 = pd.json_normalize(response.json())

df_2 = df_2.reindex(["id", "version", "owner", "createdDate", "eventStart", "description", "attachments"],axis=1)

df = pd.concat([df, df_2], axis=0, ignore_index=True).drop_duplicates("id")

df.to_csv("olog_data.csv", index=False)

df

,id,version,owner,createdDate,eventStart,description,attachments
0,34292,2,alex.cavalcante,1791606683000,1791579600000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE DETAILS.png', 'contentTyp..."
1,34291,1,vanusa.souza,1791580202000,1791580202000,### **Turno de usuários**\n\n\n\n\n- 04h23: Oc...,[]
2,34284,4,wagner.bento,1791575366000,1791570239000,#### Oscilação no feixe devido a ocorrencia de...,[{'fileName': 'Screenshot from 2026-10-09 16-2...
3,33387,2,vanusa.souza,1787257212000,1787257212000,##### **Oscilaçôes no feixe- possível influênc...,[{'fileName': 'Screenshot from 2026-08-20 16-1...
4,33338,1,alex.cavalcante,1786782933000,1786752000000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE_INDONESIA.png', 'contentT..."
5,33266,5,lucas.ransato,1786392909000,1786365268000,"**Terremoto na Colômbia - San José del Palmar,...",[{'fileName': 'Screenshot from 2026-08-10 17-1...
6,33005,6,daniel.cantarani,1785241701000,1785241701000,### Terremoto no Japão\n.\n- Por volta de 07h4...,[{'fileName': 'SismÃ³grafo terremoto Japan.PNG...
7,32807,2,wagner.bento,1784309897000,1784309897000,**TERREMOTO - EVENTO SÍSMICO DETECTADO** \n\n-...,[{'fileName': 'Screenshot from 2026-07-17 14-3...
8,32454,6,carlos.vieira,1782374640000,1782374640000,### Oscilações no feixe por Abalos Sísmicos\n'...,"[{'fileName': 'image_1782374640200.png', 'cont..."
9,32189,3,alex.cavalcante,1780908433000,1780880400000,**Oscilaçôes no feixe por abalos sísmicos**\n\...,"[{'fileName': 'quake0706.png', 'contentType': ..."


In [3]:
# !pip install openai

In [26]:
import getpass
from openai import OpenAI
import time
from datetime import datetime, timezone

In [6]:
# BASE_URL = "https://iluma.cnpem.br:4000/v1/"
# API_KEY = getpass.getpass("Insira a chave API: ")
# MODEL_NAME = "iluma"

BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
API_KEY = getpass.getpass("Insira a chave API: ")
MODEL_NAME = "models/gemini-3.5-flash-lite"

In [7]:
client = OpenAI(
    base_url=BASE_URL, 
    api_key=API_KEY
)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Olá"
        }
    ]
)

print(response.choices[0].message.content)

Olá! Como posso ajudar você hoje?


In [12]:
SYSTEM_PROMPT = """
You are a timestamp extraction engine.

TASK
Extract the event start and end times from the provided log.

INPUT

You will receive:
- Reference date (YYYY-MM-DD)
- Log text

RULES

1. Use ONLY information present in the log.
2. The reference date must always be used when a time appears without a date.
3. Assume timestamps are GMT-3 unless another timezone is explicitly specified.
4. Convert all timestamps to UTC.
5. Return timestamps in ISO 8601 format:
   YYYY-MM-DDTHH:MM:SS.000000Z

6. If an explicit time interval is found:
   - first time -> from
   - second time -> to

7. If only a single timestamp is found:
   - set "from" to that timestamp
   - set "to" to exactly 2 hours after "from"

8. If multiple timestamps are mentioned but they do not explicitly define an interval:
   - use the earliest timestamp as "from"
   - use the latest timestamp as "to"

9. If no timestamp can be determined:
   - from = null
   - to = null

10. Never return explanations.
11. Never return examples.
12. Never repeat the input.
13. Never return markdown.
14. Return only a valid JSON object.
15. All timestamps must always contain a complete ISO 8601 UTC datetime.

OUTPUT FORMAT

{
  "from": string | null,
  "to": string | null
}
"""

def get_times_from_description(log_text):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": log_text
            }
        ],
        response_format={"type": "json_object"}
    )
    
    return response.choices[0].message.content

print(get_times_from_description(df.loc[0]["description"]))

{
  "from": "2024-06-05T20:25:00.000000Z",
  "to": "2024-06-05T21:00:00.000000Z"
}


In [28]:
times_from_logs = []

for i, log in df.iterrows():
    ts = float(log["eventStart"]) / 1000

    iso = datetime.fromtimestamp(
        ts,
        tz=timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%S.000000Z")

    print(f"LOG {i:2} - {log["id"]}: Processando descrição...")
    start_time = time.time()
    response = get_times_from_description(f"Reference Date: {iso} | Description: {log["description"]}")
    delta_time = time.time() - start_time
    print(f"LOG {i:2} - {log["id"]}: Processado em {delta_time:.2f}s")
    print(response)
    time.sleep(6)
    
    times_from_logs.append(response)

LOG  0 - 34292: Processando descrição...
LOG  0 - 34292: Processado em 18.24s
{
  "from": "2026-10-09T20:25:00.000000Z",
  "to": "2026-10-09T21:00:00.000000Z"
}
LOG  1 - 34291: Processando descrição...
LOG  1 - 34291: Processado em 8.36s
{
  "from": "2026-10-09T07:23:00.000000Z",
  "to": "2026-10-09T10:41:00.000000Z"
}
LOG  2 - 34284: Processando descrição...
LOG  2 - 34284: Processado em 16.83s
{
  "from": "2026-10-09T18:13:37.000000Z",
  "to": "2026-10-09T18:23:59.000000Z"
}
LOG  3 - 33387: Processando descrição...
LOG  3 - 33387: Processado em 10.59s
{
  "from": "2026-08-20T18:00:00.000000Z",
  "to": "2026-08-20T20:00:00.000000Z"
}
LOG  4 - 33338: Processando descrição...
LOG  4 - 33338: Processado em 9.40s
{
  "from": "2026-08-15T22:00:00.000000Z",
  "to": "2026-08-16T00:00:00.000000Z"
}
LOG  5 - 33266: Processando descrição...
LOG  5 - 33266: Processado em 7.48s
{
  "from": "2026-08-10T12:34:28.000000Z",
  "to": "2026-08-10T14:34:28.000000Z"
}
LOG  6 - 33005: Processando descrição

In [31]:
import json

In [38]:
json_times_from_logs = [json.loads(x) for x in times_from_logs]
json_times_from_logs

[{'from': '2026-10-09T20:25:00.000000Z', 'to': '2026-10-09T21:00:00.000000Z'},
 {'from': '2026-10-09T07:23:00.000000Z', 'to': '2026-10-09T10:41:00.000000Z'},
 {'from': '2026-10-09T18:13:37.000000Z', 'to': '2026-10-09T18:23:59.000000Z'},
 {'from': '2026-08-20T18:00:00.000000Z', 'to': '2026-08-20T20:00:00.000000Z'},
 {'from': '2026-08-15T22:00:00.000000Z', 'to': '2026-08-16T00:00:00.000000Z'},
 {'from': '2026-08-10T12:34:28.000000Z', 'to': '2026-08-10T14:34:28.000000Z'},
 {'from': '2026-07-28T07:27:00.000000Z', 'to': '2026-07-28T09:27:00.000000Z'},
 {'from': '2026-07-17T15:06:00.000000Z', 'to': '2026-07-17T15:47:00.000000Z'},
 {'from': None, 'to': None},
 {'from': '2026-06-08T23:37:00.000000Z', 'to': '2026-06-09T01:37:00.000000Z'},
 {'from': '2026-03-21T00:22:00.000000Z', 'to': '2026-03-21T01:15:00.000000Z'},
 {'from': '2026-03-13T13:49:00.000000Z', 'to': '2026-03-13T13:55:00.000000Z'},
 {'from': '2025-09-25T03:58:00.000000Z', 'to': '2025-09-25T05:58:00.000000Z'},
 {'from': '2025-09-24T2

In [ ]:
df["from"] = [x["from"] for x in json_times_from_logs]
df["to"] = [x["to"] for x in json_times_from_logs]
df.to_csv("olog_data.csv", index=False)

df

,id,version,owner,createdDate,eventStart,description,attachments,from,to
0,34292,2,alex.cavalcante,1791606683000,1791579600000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE DETAILS.png', 'contentTyp...",2026-10-09T20:25:00.000000Z,2026-10-09T21:00:00.000000Z
1,34291,1,vanusa.souza,1791580202000,1791580202000,### **Turno de usuários**\n\n\n\n\n- 04h23: Oc...,[],2026-10-09T07:23:00.000000Z,2026-10-09T10:41:00.000000Z
2,34284,4,wagner.bento,1791575366000,1791570239000,#### Oscilação no feixe devido a ocorrencia de...,[{'fileName': 'Screenshot from 2026-10-09 16-2...,2026-10-09T18:13:37.000000Z,2026-10-09T18:23:59.000000Z
3,33387,2,vanusa.souza,1787257212000,1787257212000,##### **Oscilaçôes no feixe- possível influênc...,[{'fileName': 'Screenshot from 2026-08-20 16-1...,2026-08-20T18:00:00.000000Z,2026-08-20T20:00:00.000000Z
4,33338,1,alex.cavalcante,1786782933000,1786752000000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE_INDONESIA.png', 'contentT...",2026-08-15T22:00:00.000000Z,2026-08-16T00:00:00.000000Z
5,33266,5,lucas.ransato,1786392909000,1786365268000,"**Terremoto na Colômbia - San José del Palmar,...",[{'fileName': 'Screenshot from 2026-08-10 17-1...,2026-08-10T12:34:28.000000Z,2026-08-10T14:34:28.000000Z
6,33005,6,daniel.cantarani,1785241701000,1785241701000,### Terremoto no Japão\n.\n- Por volta de 07h4...,[{'fileName': 'SismÃ³grafo terremoto Japan.PNG...,2026-07-28T07:27:00.000000Z,2026-07-28T09:27:00.000000Z
7,32807,2,wagner.bento,1784309897000,1784309897000,**TERREMOTO - EVENTO SÍSMICO DETECTADO** \n\n-...,[{'fileName': 'Screenshot from 2026-07-17 14-3...,2026-07-17T15:06:00.000000Z,2026-07-17T15:47:00.000000Z
8,32454,6,carlos.vieira,1782374640000,1782374640000,### Oscilações no feixe por Abalos Sísmicos\n'...,"[{'fileName': 'image_1782374640200.png', 'cont...",None,None
9,32189,3,alex.cavalcante,1780908433000,1780880400000,**Oscilaçôes no feixe por abalos sísmicos**\n\...,"[{'fileName': 'quake0706.png', 'contentType': ...",2026-06-08T23:37:00.000000Z,2026-06-09T01:37:00.000000Z


In [57]:
new_df = df.copy()

new_df.loc[8, "from"] = "2024-06-26T22:46:38.000000Z"
new_df.loc[8, "to"]   = "2024-06-27T00:46:38.000000Z"

new_df = new_df.drop(35, )

new_df.loc[36, "from"] = "2025-10-31T18:51:12.000000Z"
new_df.loc[36, "to"]   = "2025-10-31T20:51:12.000000Z"

new_df.loc[38, "from"] = "2025-09-18T19:00:00.000000Z"
new_df.loc[38, "to"]   = "2025-09-18T23:00:00.000000Z"

new_df.loc[44, "from"] = "2024-12-05T19:00:00.000000Z"
new_df.loc[44, "to"]   = "2024-12-05T23:00:00.000000Z"

new_df.to_csv("olog_data.csv", index=False)

new_df

,id,version,owner,createdDate,eventStart,description,attachments,from,to
0,34292,2,alex.cavalcante,1791606683000,1791579600000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE DETAILS.png', 'contentTyp...",2026-10-09T20:25:00.000000Z,2026-10-09T21:00:00.000000Z
1,34291,1,vanusa.souza,1791580202000,1791580202000,### **Turno de usuários**\n\n\n\n\n- 04h23: Oc...,[],2026-10-09T07:23:00.000000Z,2026-10-09T10:41:00.000000Z
2,34284,4,wagner.bento,1791575366000,1791570239000,#### Oscilação no feixe devido a ocorrencia de...,[{'fileName': 'Screenshot from 2026-10-09 16-2...,2026-10-09T18:13:37.000000Z,2026-10-09T18:23:59.000000Z
3,33387,2,vanusa.souza,1787257212000,1787257212000,##### **Oscilaçôes no feixe- possível influênc...,[{'fileName': 'Screenshot from 2026-08-20 16-1...,2026-08-20T18:00:00.000000Z,2026-08-20T20:00:00.000000Z
4,33338,1,alex.cavalcante,1786782933000,1786752000000,**Oscilaçôes no feixe devido a abalos sísmicos...,"[{'fileName': 'QUAKE_INDONESIA.png', 'contentT...",2026-08-15T22:00:00.000000Z,2026-08-16T00:00:00.000000Z
5,33266,5,lucas.ransato,1786392909000,1786365268000,"**Terremoto na Colômbia - San José del Palmar,...",[{'fileName': 'Screenshot from 2026-08-10 17-1...,2026-08-10T12:34:28.000000Z,2026-08-10T14:34:28.000000Z
6,33005,6,daniel.cantarani,1785241701000,1785241701000,### Terremoto no Japão\n.\n- Por volta de 07h4...,[{'fileName': 'SismÃ³grafo terremoto Japan.PNG...,2026-07-28T07:27:00.000000Z,2026-07-28T09:27:00.000000Z
7,32807,2,wagner.bento,1784309897000,1784309897000,**TERREMOTO - EVENTO SÍSMICO DETECTADO** \n\n-...,[{'fileName': 'Screenshot from 2026-07-17 14-3...,2026-07-17T15:06:00.000000Z,2026-07-17T15:47:00.000000Z
8,32454,6,carlos.vieira,1782374640000,1782374640000,### Oscilações no feixe por Abalos Sísmicos\n'...,"[{'fileName': 'image_1782374640200.png', 'cont...",2024-06-26T22:46:38.000000Z,2024-06-27T00:46:38.000000Z
9,32189,3,alex.cavalcante,1780908433000,1780880400000,**Oscilaçôes no feixe por abalos sísmicos**\n\...,"[{'fileName': 'quake0706.png', 'contentType': ...",2026-06-08T23:37:00.000000Z,2026-06-09T01:37:00.000000Z
